In [1]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================
#
# Nesta etapa importamos as bibliotecas que serão utilizadas
# ao longo do exercício.
#
# pandas será utilizado para leitura e manipulação da base;
# numpy auxiliará em operações numéricas;
# scipy será utilizado em testes estatísticos;
# statsmodels será utilizado no cálculo de poder e tamanho
# amostral solicitado no exercício.
# ============================================================

import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.stats.power import TTestIndPower

In [3]:
# ============================================================
# 2. INSPEÇÃO E PREPARAÇÃO DOS DADOS
# ============================================================
#
# Nesta etapa carregamos o arquivo progresa_completo.csv
# e realizamos uma inspeção inicial da base.
#
# O objetivo é verificar:
# - se o arquivo foi carregado corretamente;
# - o número de observações e variáveis;
# - os nomes das variáveis disponíveis;
# - a estrutura inicial dos dados;
# - os tipos das variáveis;
# - a presença de valores ausentes nas variáveis de interesse.
# ============================================================

# ------------------------------------------------------------
# Carregamento da base
# ------------------------------------------------------------

df = pd.read_csv("progresa_completo.csv")

# ------------------------------------------------------------
# Dimensões da base
# ------------------------------------------------------------

print("=" * 60)
print("DIMENSÕES DA BASE")
print("=" * 60)

print(f"Número de observações: {df.shape[0]:,}")
print(f"Número de variáveis: {df.shape[1]}")

# ------------------------------------------------------------
# Primeiras observações
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PRIMEIRAS OBSERVAÇÕES")
print("=" * 60)

display(df.head())

# ------------------------------------------------------------
# Variáveis disponíveis
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VARIÁVEIS DISPONÍVEIS")
print("=" * 60)

print(df.columns.tolist())

# ------------------------------------------------------------
# Tipos das variáveis
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TIPOS DAS VARIÁVEIS")
print("=" * 60)

print(df.dtypes)

# ------------------------------------------------------------
# Valores ausentes nas principais variáveis do exercício
# ------------------------------------------------------------

variaveis_interesse = ["progresa", "poor", "grc"]

print("\n" + "=" * 60)
print("VALORES AUSENTES NAS VARIÁVEIS DE INTERESSE")
print("=" * 60)

for variavel in variaveis_interesse:
    if variavel in df.columns:
        print(
            f"{variavel}: "
            f"{df[variavel].isna().sum():,} valores ausentes"
        )
    else:
        print(f"{variavel}: variável não encontrada na base")

DIMENSÕES DA BASE
Número de observações: 77,250
Número de variáveis: 21

PRIMEIRAS OBSERVAÇÕES


,year,sex,indig,dist_sec,sc,grc,fam_n,min_dist,dist_cap,poor,...,hohedu,hohwag,welfare_index,hohsex,hohage,age,village,folnum,grc97,sc97
0,97,0.0,0.0,4.473,1.0,7.0,7,21.168384,21.168384,pobre,...,6,0.0,583.0,1.0,35.0,13,163,1,7,1.0
1,98,0.0,0.0,4.473,1.0,8.0,7,21.168384,21.168384,pobre,...,6,0.0,583.0,1.0,35.0,14,163,1,7,1.0
2,97,1.0,0.0,4.473,1.0,6.0,7,21.168384,21.168384,pobre,...,6,0.0,583.0,1.0,35.0,12,163,2,6,1.0
3,98,1.0,0.0,4.473,1.0,7.0,7,21.168384,21.168384,pobre,...,6,0.0,583.0,1.0,35.0,13,163,2,6,1.0
4,97,0.0,0.0,4.473,1.0,2.0,7,21.168384,21.168384,pobre,...,6,0.0,583.0,1.0,35.0,8,163,3,2,1.0



VARIÁVEIS DISPONÍVEIS
['year', 'sex', 'indig', 'dist_sec', 'sc', 'grc', 'fam_n', 'min_dist', 'dist_cap', 'poor', 'progresa', 'hohedu', 'hohwag', 'welfare_index', 'hohsex', 'hohage', 'age', 'village', 'folnum', 'grc97', 'sc97']

TIPOS DAS VARIÁVEIS
year               int64
sex              float64
indig            float64
dist_sec         float64
sc               float64
grc              float64
fam_n              int64
min_dist         float64
dist_cap         float64
poor              object
progresa          object
hohedu             int64
hohwag           float64
welfare_index    float64
hohsex           float64
hohage           float64
age                int64
village            int64
folnum             int64
grc97              int64
sc97             float64
dtype: object

VALORES AUSENTES NAS VARIÁVEIS DE INTERESSE
progresa: 0 valores ausentes
poor: 0 valores ausentes
grc: 6,549 valores ausentes


## 2. Inspeção e preparação dos dados

A base `progresa_completo.csv` contém **77.250 observações e 21 variáveis**. A inspeção inicial permitiu identificar as variáveis disponíveis, seus tipos e a presença de valores ausentes.

A variável `grc`, que será utilizada como novo desfecho neste exercício, está presente na base e possui **6.549 valores ausentes**. Portanto, a construção da amostra analítica deverá considerar apenas as observações para as quais esse resultado é observado.

Também foi constatado que a base não contém uma variável denominada `t`. A informação referente ao tratamento está disponível na variável `progresa`, armazenada como categórica (`object`). Antes de construir o indicador de tratamento, será necessário verificar como seus grupos estão codificados.

Essa inspeção preliminar é importante para evitar suposições sobre a estrutura dos dados e garantir que o recorte utilizado nas análises subsequentes seja explicitamente definido e reproduzível.

In [4]:
# ============================================================
# 3. CONSTRUÇÃO DO NOVO RECORTE UTILIZANDO GRC
# ============================================================
#
# Antes de construir a amostra analítica, verificamos como estão
# codificadas as variáveis que serão utilizadas no recorte.
#
# progresa -> identifica a condição de tratamento;
# poor     -> identifica a condição socioeconômica;
# grc      -> novo desfecho escolhido para o exercício.
#
# Essa verificação evita assumir previamente quais valores
# representam tratamento e controle.
# ============================================================

print("=" * 60)
print("CATEGORIAS DA VARIÁVEL PROGRESA")
print("=" * 60)

print(df["progresa"].value_counts(dropna=False))


print("\n" + "=" * 60)
print("CATEGORIAS DA VARIÁVEL POOR")
print("=" * 60)

print(df["poor"].value_counts(dropna=False))


print("\n" + "=" * 60)
print("DISTRIBUIÇÃO DA VARIÁVEL GRC")
print("=" * 60)

print(df["grc"].describe())


print("\n" + "=" * 60)
print("VALORES AUSENTES EM GRC")
print("=" * 60)

n_ausentes_grc = df["grc"].isna().sum()
perc_ausentes_grc = df["grc"].isna().mean() * 100

print(f"Número de valores ausentes: {n_ausentes_grc:,}")
print(f"Percentual de valores ausentes: {perc_ausentes_grc:.2f}%")

CATEGORIAS DA VARIÁVEL PROGRESA
progresa
basal    47560
0        29690
Name: count, dtype: int64

CATEGORIAS DA VARIÁVEL POOR
poor
pobre       65392
no pobre    11858
Name: count, dtype: int64

DISTRIBUIÇÃO DA VARIÁVEL GRC
count    70701.000000
mean         3.963537
std          2.499063
min          0.000000
25%          2.000000
50%          4.000000
75%          6.000000
max         14.000000
Name: grc, dtype: float64

VALORES AUSENTES EM GRC
Número de valores ausentes: 6,549
Percentual de valores ausentes: 8.48%


In [5]:
# ============================================================
# 3.1 VERIFICAÇÃO DAS VARIÁVEIS PARA DEFINIÇÃO DO RECORTE
# ============================================================
#
# Nesta etapa verificamos a distribuição conjunta das principais
# variáveis envolvidas na definição da amostra analítica.
#
# O objetivo é identificar corretamente:
# - os períodos disponíveis;
# - a condição de elegibilidade (poor);
# - a variável associada ao tratamento;
# - a disponibilidade do novo desfecho (grc).
# ============================================================

print("=" * 60)
print("DISTRIBUIÇÃO POR ANO")
print("=" * 60)
print(df["year"].value_counts().sort_index())

print("\n" + "=" * 60)
print("DISTRIBUIÇÃO DA VARIÁVEL SC")
print("=" * 60)
print(df["sc"].value_counts(dropna=False).sort_index())

print("\n" + "=" * 60)
print("PROGRESA × SC")
print("=" * 60)
print(pd.crosstab(df["progresa"], df["sc"], margins=True))

print("\n" + "=" * 60)
print("POOR × SC")
print("=" * 60)
print(pd.crosstab(df["poor"], df["sc"], margins=True))

print("\n" + "=" * 60)
print("OBSERVAÇÕES VÁLIDAS DE GRC POR ANO")
print("=" * 60)
print(
    df.groupby("year")["grc"]
      .agg(["count", "mean", "std"])
      .round(3)
)

DISTRIBUIÇÃO POR ANO
year
97    38625
98    38625
Name: count, dtype: int64

DISTRIBUIÇÃO DA VARIÁVEL SC
sc
0.0    12396
1.0    56401
NaN     8453
Name: count, dtype: int64

PROGRESA × SC
sc          0.0    1.0    All
progresa                     
0          5142  21210  26352
basal      7254  35191  42445
All       12396  56401  68797

POOR × SC
sc          0.0    1.0    All
poor                         
no pobre   2206   8219  10425
pobre     10190  48182  58372
All       12396  56401  68797

OBSERVAÇÕES VÁLIDAS DE GRC POR ANO
      count   mean    std
year                     
97    38625  3.705  2.572
98    32076  4.274  2.371


### Análise da estrutura das variáveis utilizadas no recorte

A base contém 77.250 observações, distribuídas igualmente entre os anos de 1997 e 1998. A variável `poor` distingue as observações classificadas como `pobre` e `no pobre`, enquanto `progresa` apresenta as categorias `basal` e `0`.

A variável `grc`, selecionada como novo desfecho, possui 70.701 observações válidas, com média de aproximadamente 3,96, mediana igual a 4 e valores entre 0 e 14. Foram identificados 6.549 valores ausentes, correspondentes a 8,48% da base.

A análise por período mostra que `grc` não apresenta valores ausentes em 1997, quando estão disponíveis 38.625 observações. Em 1998, entretanto, estão disponíveis 32.076 observações válidas. Assim, a disponibilidade do novo desfecho varia entre os períodos e deverá ser considerada na construção da amostra analítica.

In [6]:
# ============================================================
# 4. DEFINIÇÃO DO TRATAMENTO E DA ELEGIBILIDADE
# ============================================================
#
# Seguindo a definição utilizada na aula:
#
# T = 1 -> criança pertence a um vilarejo sorteado
#          para receber o PROGRESA
# T = 0 -> criança pertence a um vilarejo de controle
#
# A variável "poor" identifica as famílias elegíveis
# ao programa segundo o critério de renda.
#
# elegivel = 1 -> família classificada como "pobre"
# elegivel = 0 -> família classificada como "no pobre"
# ============================================================

# Criação da variável de tratamento
df["T"] = (df["progresa"] == "basal").astype(int)

# Criação da variável de elegibilidade
df["elegivel"] = (df["poor"] == "pobre").astype(int)


# ------------------------------------------------------------
# Verificação das variáveis criadas
# ------------------------------------------------------------

print("=" * 60)
print("DISTRIBUIÇÃO DA VARIÁVEL DE TRATAMENTO (T)")
print("=" * 60)
print(df["T"].value_counts().sort_index())

print("\n" + "=" * 60)
print("DISTRIBUIÇÃO DA VARIÁVEL DE ELEGIBILIDADE")
print("=" * 60)
print(df["elegivel"].value_counts().sort_index())


# ------------------------------------------------------------
# Conferência da correspondência com as variáveis originais
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PROGRESA x T")
print("=" * 60)
print(pd.crosstab(df["progresa"], df["T"]))

print("\n" + "=" * 60)
print("POOR x ELEGÍVEL")
print("=" * 60)
print(pd.crosstab(df["poor"], df["elegivel"]))

DISTRIBUIÇÃO DA VARIÁVEL DE TRATAMENTO (T)
T
0    29690
1    47560
Name: count, dtype: int64

DISTRIBUIÇÃO DA VARIÁVEL DE ELEGIBILIDADE
elegivel
0    11858
1    65392
Name: count, dtype: int64

PROGRESA x T
T             0      1
progresa              
0         29690      0
basal         0  47560

POOR x ELEGÍVEL
elegivel      0      1
poor                  
no pobre  11858      0
pobre         0  65392


### Definição do tratamento e da elegibilidade

Seguindo a estrutura do experimento apresentada em aula, foi criada a variável binária `T` para representar a designação ao tratamento. O valor `T = 1` identifica crianças residentes em vilarejos sorteados para receber o PROGRESA (`progresa = basal`), enquanto `T = 0` identifica aquelas residentes nos vilarejos de controle.

Também foi criada a variável `elegivel`, que identifica as famílias pertencentes ao público elegível ao programa segundo o critério de renda. Assim, famílias classificadas como `pobre` receberam `elegivel = 1`, enquanto as classificadas como `no pobre` receberam `elegivel = 0`.

Essa distinção é importante porque a unidade de aleatorização do experimento é o vilarejo, enquanto a elegibilidade ao programa é determinada no nível da família.

In [7]:
# ============================================================
# 5. CONSTRUÇÃO DA AMOSTRA DE LINHA DE BASE
# ============================================================
#
# Para avaliar o balanceamento entre os grupos de tratamento
# e controle, utilizamos apenas as observações:
#
# 1. referentes ao ano de 1997 (linha de base);
# 2. pertencentes às famílias elegíveis ao PROGRESA.
#
# Esse recorte permite comparar tratamento e controle antes
# da implementação do programa.
# ============================================================

base97 = df[
    (df["year"] == 97) &
    (df["elegivel"] == 1)
].copy()


# ------------------------------------------------------------
# Dimensões da amostra de linha de base
# ------------------------------------------------------------

print("=" * 60)
print("AMOSTRA DE LINHA DE BASE — 1997")
print("=" * 60)

print(f"Número de observações: {base97.shape[0]:,}")
print(f"Número de variáveis: {base97.shape[1]}")


# ------------------------------------------------------------
# Distribuição entre tratamento e controle
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DISTRIBUIÇÃO ENTRE TRATAMENTO E CONTROLE")
print("=" * 60)

print(base97["T"].value_counts().sort_index())


# ------------------------------------------------------------
# Verificação do recorte realizado
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VERIFICAÇÃO DO RECORTE")
print("=" * 60)

print("Anos presentes:", base97["year"].unique())
print("Elegibilidade:", base97["elegivel"].unique())


# ------------------------------------------------------------
# Valores válidos da variável de resultado GRC
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DISPONIBILIDADE DA VARIÁVEL GRC")
print("=" * 60)

print(f"Observações totais em base97: {len(base97):,}")
print(f"Observações com GRC válido: {base97['grc'].notna().sum():,}")
print(f"Valores ausentes em GRC: {base97['grc'].isna().sum():,}")

AMOSTRA DE LINHA DE BASE — 1997
Número de observações: 32,696
Número de variáveis: 23

DISTRIBUIÇÃO ENTRE TRATAMENTO E CONTROLE
T
0    12474
1    20222
Name: count, dtype: int64

VERIFICAÇÃO DO RECORTE
Anos presentes: [97]
Elegibilidade: [1]

DISPONIBILIDADE DA VARIÁVEL GRC
Observações totais em base97: 32,696
Observações com GRC válido: 32,696
Valores ausentes em GRC: 0


### Construção da amostra de linha de base

Para avaliar a comparabilidade inicial entre os grupos de tratamento e controle, a análise foi restringida às famílias elegíveis ao PROGRESA observadas em 1997, período correspondente à linha de base do experimento.

O recorte resultou em **32.696 observações**, das quais **20.222 pertencem ao grupo de tratamento** e **12.474 ao grupo de controle**. As verificações realizadas confirmaram que todas as observações selecionadas correspondem ao ano de 1997 e a famílias classificadas como elegíveis ao programa.

Também foi verificada a disponibilidade da variável `grc`, escolhida como novo resultado para o exercício. Não foram identificados valores ausentes nessa variável na amostra de linha de base, de modo que as **32.696 observações permanecem disponíveis para as análises subsequentes**.

Esse recorte será utilizado para verificar o balanceamento entre os grupos antes da implementação do programa, permitindo avaliar se tratamento e controle apresentavam características observáveis semelhantes no período pré-tratamento.

In [11]:
# ============================================================
# 6. TABELA DE BALANCEAMENTO
# ============================================================
#
# A tabela de balanceamento compara características observáveis
# dos grupos de tratamento e controle na linha de base (1997).
#
# Para cada variável, calculamos:
#
# 1. média do grupo de controle;
# 2. média do grupo de tratamento;
# 3. diferença entre as médias (Tratamento - Controle);
# 4. erro-padrão da diferença entre as médias;
# 5. p-valor do teste t de diferença de médias.
#
# Os nomes originais das variáveis são preservados na base.
# Para facilitar a interpretação da tabela, utilizamos rótulos
# descritivos em português apenas na apresentação dos resultados.
#
# A análise utiliza a amostra base97, construída anteriormente,
# contendo apenas famílias elegíveis observadas em 1997.
# ============================================================


# ------------------------------------------------------------
# Variáveis utilizadas na tabela de balanceamento
# ------------------------------------------------------------

variaveis_balanceamento = [
    "age",
    "sex",
    "indig",
    "hohedu",
    "hohwag",
    "welfare_index",
    "hohsex",
    "hohage"
]


# ------------------------------------------------------------
# Rótulos em português para apresentação
# ------------------------------------------------------------

rotulos_variaveis = {
    "age": "Idade",
    "sex": "Sexo",
    "indig": "Indígena",
    "hohedu": "Escolaridade do chefe do domicílio",
    "hohwag": "Rendimento mensal do chefe do domicílio",
    "welfare_index": "Índice de bem-estar",
    "hohsex": "Sexo do chefe do domicílio",
    "hohage": "Idade do chefe do domicílio"
}


# ------------------------------------------------------------
# Construção da tabela
# ------------------------------------------------------------

resultados_balanceamento = []

for var in variaveis_balanceamento:

    # Observações válidas de cada grupo
    controle = base97.loc[
        base97["T"] == 0,
        var
    ].dropna()

    tratamento = base97.loc[
        base97["T"] == 1,
        var
    ].dropna()

    # Médias dos grupos
    media_controle = controle.mean()
    media_tratamento = tratamento.mean()

    # Diferença entre as médias
    # Tratamento - Controle
    diferenca = media_tratamento - media_controle

    # Erro-padrão da diferença entre as médias
    erro_padrao = np.sqrt(
        tratamento.var(ddof=1) / len(tratamento)
        +
        controle.var(ddof=1) / len(controle)
    )

    # Teste t de Welch para diferença de médias
    teste = stats.ttest_ind(
        tratamento,
        controle,
        equal_var=False,
        nan_policy="omit"
    )

    # Armazenamento dos resultados
    resultados_balanceamento.append({
        "Variável": rotulos_variaveis[var],
        "Controle": media_controle,
        "Tratamento": media_tratamento,
        "Diferença": diferenca,
        "Erro-padrão": erro_padrao,
        "p-valor": teste.pvalue
    })


# ------------------------------------------------------------
# Organização dos resultados
# ------------------------------------------------------------

tabela_balanceamento = pd.DataFrame(
    resultados_balanceamento
)


# ------------------------------------------------------------
# Arredondamento apenas para apresentação
# ------------------------------------------------------------

colunas_numericas = [
    "Controle",
    "Tratamento",
    "Diferença",
    "Erro-padrão",
    "p-valor"
]

tabela_balanceamento[colunas_numericas] = (
    tabela_balanceamento[colunas_numericas]
    .round(4)
)


# ------------------------------------------------------------
# Apresentação da tabela
# ------------------------------------------------------------

print("=" * 85)
print("TABELA DE BALANCEAMENTO — LINHA DE BASE (1997)")
print("=" * 85)

display(tabela_balanceamento)

TABELA DE BALANCEAMENTO — LINHA DE BASE (1997)


,Variável,Controle,Tratamento,Diferença,Erro-padrão,p-valor
0,Idade,10.7420,10.7170,-0.0250,0.0353,0.4784
1,Sexo,0.5051,0.5193,0.0143,0.0057,0.0122
2,Indígena,0.3322,0.3260,-0.0062,0.0054,0.2459
3,Escolaridade do chefe do domicílio,2.5903,2.6631,0.0728,0.0284,0.0104
4,Rendimento mensal do chefe do domicílio,573.1636,544.3395,-28.8240,8.0245,0.0003
5,Índice de bem-estar,659.5791,655.4284,-4.1507,1.3098,0.0015
6,Sexo do chefe do domicílio,0.9229,0.9247,0.0017,0.0030,0.5721
7,Idade do chefe do domicílio,44.2769,43.6488,-0.6281,0.1328,0.0000


### 6. Análise do balanceamento na linha de base

A tabela de balanceamento compara as características observáveis dos grupos de tratamento e controle na linha de base, correspondente ao ano de 1997. O objetivo é verificar se os dois grupos apresentavam características semelhantes antes da implementação do programa, condição relevante para a interpretação posterior dos efeitos do tratamento.

Os resultados mostram que algumas características apresentam diferenças pequenas e não estatisticamente significativas entre os grupos. Esse é o caso da **idade**, da condição de **indígena** e do **sexo do chefe do domicílio**, cujos p-valores são superiores a 0,05.

Por outro lado, são observadas diferenças estatisticamente significativas em algumas características. O grupo de tratamento apresenta proporção ligeiramente maior na variável **sexo** (0,5193 contra 0,5051; p = 0,0122) e maior **escolaridade do chefe do domicílio** (2,6631 contra 2,5903; p = 0,0104). Em sentido contrário, o grupo de tratamento apresenta menor **renda do chefe do domicílio** (544,34 contra 573,16; p = 0,0003), menor **índice de bem-estar** (655,43 contra 659,58; p = 0,0015) e menor **idade do chefe do domicílio** (43,65 contra 44,28 anos; p < 0,001).

Portanto, a tabela indica que os grupos de tratamento e controle **não estão perfeitamente balanceados em todas as características observáveis na linha de base**. Embora algumas diferenças sejam pequenas em magnitude, a existência de diferenças estatisticamente significativas deve ser considerada na interpretação dos resultados posteriores. Em particular, o balanceamento não deve ser avaliado exclusivamente pela significância estatística, mas também pela magnitude e pela relevância substantiva das diferenças observadas.

In [12]:
# ============================================================
# 7. ESTIMAÇÃO DO EFEITO DO PROGRESA SOBRE A SÉRIE ESCOLAR
# ============================================================
#
# Nesta etapa estimamos o efeito do PROGRESA sobre a variável
# de resultado grc (série escolar).
#
# Como o tratamento foi definido pela alocação das localidades
# ao PROGRESA, comparamos em 1998 a média de grc entre:
#
# T = 1 -> grupo de tratamento
# T = 0 -> grupo de controle
#
# Mantemos apenas:
# - famílias elegíveis ao programa;
# - observações referentes a 1998;
# - crianças com valor válido para grc.
#
# O efeito é calculado como:
#
# média do tratamento - média do controle
#
# Também calculamos:
# - erro-padrão da diferença;
# - teste t de Welch;
# - intervalo de confiança de 95%.
# ============================================================


# ------------------------------------------------------------
# Construção da amostra pós-tratamento
# ------------------------------------------------------------

base98 = df[
    (df["year"] == 98) &
    (df["elegivel"] == 1) &
    (df["grc"].notna())
].copy()


# ------------------------------------------------------------
# Separação entre tratamento e controle
# ------------------------------------------------------------

grc_controle = base98.loc[
    base98["T"] == 0,
    "grc"
].dropna()

grc_tratamento = base98.loc[
    base98["T"] == 1,
    "grc"
].dropna()


# ------------------------------------------------------------
# Estatísticas descritivas
# ------------------------------------------------------------

n_controle = len(grc_controle)
n_tratamento = len(grc_tratamento)

media_controle = grc_controle.mean()
media_tratamento = grc_tratamento.mean()

dp_controle = grc_controle.std(ddof=1)
dp_tratamento = grc_tratamento.std(ddof=1)


# ------------------------------------------------------------
# Estimativa do efeito
# ------------------------------------------------------------

efeito = media_tratamento - media_controle


# ------------------------------------------------------------
# Erro-padrão da diferença
# ------------------------------------------------------------

erro_padrao = np.sqrt(
    (dp_tratamento**2 / n_tratamento) +
    (dp_controle**2 / n_controle)
)


# ------------------------------------------------------------
# Teste t de Welch
# ------------------------------------------------------------

teste = stats.ttest_ind(
    grc_tratamento,
    grc_controle,
    equal_var=False
)


# ------------------------------------------------------------
# Intervalo de confiança de 95%
# ------------------------------------------------------------

ic_inferior = efeito - 1.96 * erro_padrao
ic_superior = efeito + 1.96 * erro_padrao


# ------------------------------------------------------------
# Organização dos resultados
# ------------------------------------------------------------

resultado_efeito = pd.DataFrame({
    "Indicador": [
        "N - Controle",
        "N - Tratamento",
        "Média - Controle",
        "Média - Tratamento",
        "Efeito estimado",
        "Erro-padrão",
        "IC 95% - Limite inferior",
        "IC 95% - Limite superior",
        "p-valor"
    ],
    "Valor": [
        n_controle,
        n_tratamento,
        media_controle,
        media_tratamento,
        efeito,
        erro_padrao,
        ic_inferior,
        ic_superior,
        teste.pvalue
    ]
})


# ------------------------------------------------------------
# Apresentação dos resultados
# ------------------------------------------------------------

resultado_efeito["Valor"] = resultado_efeito["Valor"].round(4)

print("=" * 75)
print("EFEITO DO PROGRESA SOBRE A SÉRIE ESCOLAR (GRC) — 1998")
print("=" * 75)

display(resultado_efeito)

EFEITO DO PROGRESA SOBRE A SÉRIE ESCOLAR (GRC) — 1998


,Indicador,Valor
0,N - Controle,10421.0000
1,N - Tratamento,17011.0000
2,Média - Controle,4.1304
3,Média - Tratamento,4.1410
4,Efeito estimado,0.0106
5,Erro-padrão,0.0290
6,IC 95% - Limite inferior,-0.0463
7,IC 95% - Limite superior,0.0674
8,p-valor,0.7159


### 7. Efeito do PROGRESA sobre a série escolar em 1998

Para avaliar o efeito do PROGRESA sobre a **série escolar (`grc`)**, foram comparadas as médias dos grupos de tratamento e controle em 1998, considerando apenas as famílias elegíveis ao programa e as observações com informação válida para a variável de resultado.

A amostra utilizada nesta etapa contém **10.421 observações no grupo de controle** e **17.011 no grupo de tratamento**. A série escolar média foi de **4,1304 no grupo de controle** e **4,1410 no grupo de tratamento**, resultando em uma diferença estimada de **0,0106 série escolar** em favor do grupo de tratamento.

Entretanto, essa diferença é pequena em magnitude e **não é estatisticamente significativa**. O erro-padrão estimado foi de **0,0290**, com intervalo de confiança de 95% entre **-0,0463 e 0,0674** e **p-valor de 0,7159**. Como o intervalo de confiança inclui zero e o p-valor é superior ao nível convencional de significância de 5%, não se rejeita a hipótese nula de ausência de diferença entre os grupos.

Assim, para a variável `grc`, **não foram encontradas evidências estatísticas de que o PROGRESA tenha produzido efeito sobre a série escolar média em 1998**, considerando a comparação realizada nesta amostra. Esse resultado deve ser interpretado como ausência de evidência de efeito nesta especificação, e não como demonstração de que o efeito do programa seja necessariamente igual a zero.

In [13]:
# ============================================================
# 8. CÁLCULO DO TAMANHO AMOSTRAL PARA PODER DE 90%
# ============================================================
#
# Nesta etapa calculamos o tamanho amostral necessário para
# detectar um efeito de 0,10 desvio-padrão (0,10σ) sobre a
# variável de resultado, com poder estatístico de 90%.
#
# Utilizamos a classe TTestIndPower, do statsmodels, considerando:
#
# effect_size = 0.10  -> efeito padronizado de 0,10σ
# alpha = 0.05        -> nível de significância de 5%
# power = 0.90        -> poder estatístico de 90%
#
# Inicialmente, o cálculo considera grupos de tratamento e
# controle de mesmo tamanho (ratio = 1).
# ============================================================


# ------------------------------------------------------------
# Parâmetros do cálculo
# ------------------------------------------------------------

effect_size = 0.10
alpha = 0.05
power = 0.90
ratio = 1.0


# ------------------------------------------------------------
# Objeto para análise de poder
# ------------------------------------------------------------

analise_poder = TTestIndPower()


# ------------------------------------------------------------
# Tamanho amostral necessário
# ------------------------------------------------------------

n_por_grupo = analise_poder.solve_power(
    effect_size=effect_size,
    nobs1=None,
    alpha=alpha,
    power=power,
    ratio=ratio,
    alternative="two-sided"
)


# O resultado pode não ser inteiro.
# Como não é possível observar uma fração de indivíduo,
# arredondamos sempre para cima.

n_por_grupo_arredondado = int(np.ceil(n_por_grupo))

n_total = 2 * n_por_grupo_arredondado


# ------------------------------------------------------------
# Organização dos resultados
# ------------------------------------------------------------

resultado_poder = pd.DataFrame({
    "Parâmetro": [
        "Efeito padronizado",
        "Nível de significância",
        "Poder estatístico",
        "Razão Tratamento/Controle",
        "N necessário por grupo",
        "N total necessário"
    ],
    "Valor": [
        effect_size,
        alpha,
        power,
        ratio,
        n_por_grupo_arredondado,
        n_total
    ]
})


# ------------------------------------------------------------
# Apresentação
# ------------------------------------------------------------

print("=" * 75)
print("CÁLCULO DO TAMANHO AMOSTRAL — PODER DE 90%")
print("=" * 75)

display(resultado_poder)

CÁLCULO DO TAMANHO AMOSTRAL — PODER DE 90%


,Parâmetro,Valor
0,Efeito padronizado,0.10
1,Nível de significância,0.05
2,Poder estatístico,0.90
3,Razão Tratamento/Controle,1.00
4,N necessário por grupo,2103.00
5,N total necessário,4206.00


In [14]:
# ============================================================
# 9. ANÁLISE DE PODER COM A PROPORÇÃO OBSERVADA NA AMOSTRA
# ============================================================
#
# No Item 8, o cálculo do tamanho amostral considerou grupos
# de tratamento e controle de mesmo tamanho (ratio = 1).
#
# Entretanto, a amostra utilizada na análise de 1998 apresenta
# números diferentes de observações nos dois grupos:
#
# Controle   = 10.421
# Tratamento = 17.011
#
# Nesta etapa, repetimos o cálculo utilizando a razão observada
# entre tratamento e controle.
#
# Mantemos:
#
# effect_size = 0.10  -> efeito de 0,10σ
# alpha = 0.05        -> nível de significância de 5%
# power = 0.90        -> poder estatístico de 90%
# ============================================================


# ------------------------------------------------------------
# Razão observada entre tratamento e controle
# ------------------------------------------------------------

ratio_observado = n_tratamento / n_controle


# ------------------------------------------------------------
# Tamanho necessário do grupo de controle
# ------------------------------------------------------------

n_controle_necessario = analise_poder.solve_power(
    effect_size=effect_size,
    nobs1=None,
    alpha=alpha,
    power=power,
    ratio=ratio_observado,
    alternative="two-sided"
)


# ------------------------------------------------------------
# Arredondamento para números inteiros
# ------------------------------------------------------------

n_controle_necessario = int(
    np.ceil(n_controle_necessario)
)

n_tratamento_necessario = int(
    np.ceil(n_controle_necessario * ratio_observado)
)

n_total_necessario_observado = (
    n_controle_necessario +
    n_tratamento_necessario
)


# ------------------------------------------------------------
# Comparação com a amostra efetivamente disponível
# ------------------------------------------------------------

n_total_observado = n_controle + n_tratamento


# ------------------------------------------------------------
# Organização dos resultados
# ------------------------------------------------------------

resultado_poder_observado = pd.DataFrame({
    "Indicador": [
        "Razão observada Tratamento/Controle",
        "N necessário - Controle",
        "N necessário - Tratamento",
        "N total necessário",
        "N observado - Controle",
        "N observado - Tratamento",
        "N total observado"
    ],
    "Valor": [
        ratio_observado,
        n_controle_necessario,
        n_tratamento_necessario,
        n_total_necessario_observado,
        n_controle,
        n_tratamento,
        n_total_observado
    ]
})


# ------------------------------------------------------------
# Apresentação
# ------------------------------------------------------------

resultado_poder_observado["Valor"] = (
    resultado_poder_observado["Valor"].round(2)
)

print("=" * 75)
print("ANÁLISE DE PODER — PROPORÇÃO OBSERVADA NA AMOSTRA")
print("=" * 75)

display(resultado_poder_observado)

ANÁLISE DE PODER — PROPORÇÃO OBSERVADA NA AMOSTRA


,Indicador,Valor
0,Razão observada Tratamento/Controle,1.63
1,N necessário - Controle,1696.00
2,N necessário - Tratamento,2769.00
3,N total necessário,4465.00
4,N observado - Controle,10421.00
5,N observado - Tratamento,17011.00
6,N total observado,27432.00


### 8 e 9. Poder estatístico e tamanho amostral

Para avaliar a capacidade da análise de detectar um efeito de pequena magnitude, foi calculado o tamanho amostral necessário para identificar um efeito padronizado de **0,10σ**, adotando nível de significância de **5%** e poder estatístico de **90%**.

Inicialmente, considerando grupos de tratamento e controle de mesmo tamanho, o cálculo indicou a necessidade de **2.103 observações por grupo**, correspondendo a uma amostra total de **4.206 observações**.

Como a amostra efetivamente utilizada em 1998 não apresenta grupos de mesmo tamanho, o cálculo foi repetido considerando a proporção observada entre tratamento e controle, de aproximadamente **1,63**. Nesse cenário, seriam necessárias **1.696 observações no grupo de controle** e **2.769 no grupo de tratamento**, totalizando **4.465 observações**.

A amostra disponível para a análise contém **10.421 observações no grupo de controle** e **17.011 no grupo de tratamento**, totalizando **27.432 observações**. Portanto, o número de observações disponíveis é substancialmente superior ao tamanho amostral estimado como necessário para detectar um efeito de 0,10σ com poder de 90%.

Esse resultado complementa a análise do efeito do PROGRESA sobre a série escolar (`grc`). No Item 7, a diferença estimada entre tratamento e controle foi de apenas **0,0106 série escolar**, com **p-valor de 0,7159** e intervalo de confiança de 95% incluindo zero. Considerando que a amostra disponível supera amplamente o tamanho calculado para detectar um efeito padronizado de 0,10σ, a ausência de significância estatística observada não parece decorrer simplesmente de insuficiência de tamanho amostral para detectar um efeito dessa magnitude.

Em conjunto, os resultados indicam que, para a variável `grc` e para a especificação adotada, **não foram encontradas evidências de um efeito do PROGRESA de magnitude igual ou superior ao efeito padronizado de 0,10σ utilizado no cálculo de poder**.

## Considerações finais

Este exercício avaliou o efeito do PROGRESA sobre a série escolar (`grc`), utilizando um recorte da base `progresa_completo.csv`. A análise considerou famílias elegíveis e utilizou informações da linha de base de 1997 para caracterizar os grupos de tratamento e controle.

A tabela de balanceamento mostrou que algumas características observáveis apresentaram diferenças estatisticamente significativas entre os grupos na linha de base. Esse resultado recomenda cautela na interpretação de uma comparação simples de médias como estimativa causal, uma vez que tratamento e controle não se mostraram perfeitamente balanceados em todas as características analisadas.

Para o resultado educacional, a diferença estimada entre os grupos em 1998 foi de aproximadamente **0,011 série escolar**, com erro-padrão de **0,029** e p-valor de **0,716**. Assim, nesta especificação, não encontramos evidência estatística de diferença entre tratamento e controle para a variável `grc`.

Por fim, a análise de poder realizada com `statsmodels` indicou que, para detectar um efeito padronizado de **0,10σ**, com nível de significância de 5% e poder estatístico de 90%, seriam necessárias aproximadamente **4.206 observações** sob uma alocação equilibrada entre tratamento e controle. Considerando a proporção observada na amostra, de aproximadamente **1,63 tratado para cada controle**, o tamanho amostral necessário seria de cerca de **4.465 observações**. A amostra utilizada na análise contém **27.432 observações**, número substancialmente superior ao requisito calculado.

Em conjunto, os resultados mostram que a ausência de significância estatística do efeito estimado sobre `grc` não parece decorrer simplesmente de insuficiência do tamanho amostral para detectar um efeito padronizado de 0,10σ. Ao mesmo tempo, as diferenças observadas no balanceamento indicam que a interpretação causal dos resultados deve considerar cuidadosamente a comparabilidade entre os grupos.